# Notebook 33 — NDJF climatological setting for the JPCZ conceptual diagram

This notebook makes **seasonal background-state** figures, not event composites. It uses all hourly ERA5 values from November, December, January, and February in 2000–2025. The output is two presentation-ready maps:

1. **Large scale:** absolute NDJF mean sea-level pressure (MSLP), actual 925-hPa winds, and the automatically located Siberian High and Aleutian Low.
2. **Regional:** wider terrain/elevation relief around the Changbai Mountains and Japan/East Sea, Sea-of-Japan mean SST shading, 2-m air-temperature contours, and actual NDJF mean 925-hPa wind vectors.

The monthly ERA5 reductions are checkpointed individually in Google Drive. If Colab disconnects, rerun the build cell: it resumes from completed months and does not recompute them. The Mapzen/AWS Terrain Tiles background is used only for presentation relief; ERA5 supplies every atmospheric and SST field.

The two maps should be presented as complementary scales: the large-scale map establishes the Siberian High–Aleutian Low pressure setting; the regional map shows how its low-level flow encounters the Changbai/coastal terrain and crosses the Japan/East Sea.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git'
BRANCH = 'codex/notebook16-pcolormesh'
REPO_DIR = '/content/JPCZcatalog'
FORCE_REFRESH_REPO = True

from google.colab import drive
# A runtime reset can leave a stale DriveFS process. Retry once with a forced remount.
try:
    drive.mount('/content/drive')
except ValueError:
    print('Initial Google Drive mount failed; retrying with force_remount=True…')
    drive.mount('/content/drive', force_remount=True)
DRIVE_ROOT = Path('/content/drive/MyDrive/JPCZcatalog_outputs')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if FORCE_REFRESH_REPO and Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)
if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements-colab.txt'], check=True)
# Explicit reinstall/verification: Cartopy is required for both map panels after a runtime reset.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'cartopy'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR], check=True)
import cartopy
print('Cartopy ready:', cartopy.__version__)
os.chdir(REPO_DIR)
if f'{REPO_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/src')
print('Repository branch:', BRANCH)
print('Drive root:', DRIVE_ROOT)

In [ ]:
from pathlib import Path
import shutil

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
import matplotlib.patheffects as pe
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

from jpcz_catalog.config import BoundingBox
from jpcz_catalog.concept_climatology import (
    ALEUTIAN_LOW_BOX, LARGE_SCALE_DOMAIN, REGIONAL_DOMAIN, SEA_OF_JAPAN_DISPLAY_BOX,
    SIBERIAN_HIGH_BOX, build_terrain_relief, find_pressure_center,
    load_or_build_ndjf_concept_climatology, subset_domain,
)

CLIMATOLOGY_YEARS = tuple(range(2000, 2026))
CLIMATOLOGY_MONTHS = (11, 12, 1, 2)
# Four-day remote reads are safe because the helper now loads one variable at a time and selects 925 hPa first.
# This cuts cloud requests by about 75% compared with one-day reads.
TIME_REDUCTION_CHUNK_DAYS = 4
RUN_TAG = f'ndjf_{min(CLIMATOLOGY_YEARS)}_{max(CLIMATOLOGY_YEARS)}_months_11_12_01_02'
# Presentation domain: includes both the Siberian High and the Aleutian Low while retaining the Sea of Japan focus.
CONCEPT_MAP_DOMAIN = BoundingBox(lon_min=90.0, lon_max=180.0, lat_min=30.0, lat_max=60.0)

DRIVE_CACHE_DIR = DRIVE_ROOT / 'ndjf_concept_climatology' / RUN_TAG
DRIVE_FIGURE_DIR = DRIVE_ROOT / 'ndjf_climatological_context_figures'
LOCAL_WORK_DIR = Path('/content/jpcz_ndjf_context_work')
LOCAL_FIGURE_DIR = Path('outputs/verification/ndjf_climatological_context_figures')
for directory in [DRIVE_CACHE_DIR, DRIVE_FIGURE_DIR, LOCAL_WORK_DIR, LOCAL_FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MEAN_CACHE_PATH = DRIVE_CACHE_DIR / f'{RUN_TAG}_mean.nc'
INVENTORY_PATH = DRIVE_CACHE_DIR / f'{RUN_TAG}_inventory.csv'
CENTER_SUMMARY_PATH = DRIVE_FIGURE_DIR / f'{RUN_TAG}_pressure_centers.csv'
LARGE_SCALE_FIGURE_PATH = DRIVE_FIGURE_DIR / f'{RUN_TAG}_large_scale_mslp_925wind.png'
REGIONAL_FIGURE_PATH = DRIVE_FIGURE_DIR / f'{RUN_TAG}_regional_terrain_sst_t2m_925wind.png'

print('Climatology period:', f'NDJF {min(CLIMATOLOGY_YEARS)}–{max(CLIMATOLOGY_YEARS)}')
print('Monthly checkpoints:', DRIVE_CACHE_DIR)

## Build or resume the NDJF mean fields

The first run may take time because it reduces ERA5 in 4-day pieces. Each finished calendar month is immediately saved in Drive. When the cell ends with `104 complete; 0 pending`, it also writes one compact final mean-field file.

In [ ]:
# If this name is missing, the Colab runtime reset: rerun the setup cell and the imports/configuration cell above.
if 'load_or_build_ndjf_concept_climatology' not in globals():
    raise RuntimeError('Colab runtime reset. Run the setup cell, then the imports/configuration cell, and rerun this build cell.')

if MEAN_CACHE_PATH.exists():
    # A completed mean is sufficient for rerendering figures; do not reopen 104 monthly checkpoints.
    ndjf_mean = xr.open_dataset(MEAN_CACHE_PATH).load()
    inventory = pd.read_csv(INVENTORY_PATH) if INVENTORY_PATH.exists() else pd.DataFrame({'status': ['complete'] * 104})
    print('Loaded completed NDJF mean from Drive:', MEAN_CACHE_PATH)
else:
    ndjf_mean, inventory = load_or_build_ndjf_concept_climatology(
        cache_dir=DRIVE_CACHE_DIR,
        local_work_dir=LOCAL_WORK_DIR,
        years=CLIMATOLOGY_YEARS,
        months=CLIMATOLOGY_MONTHS,
        large_domain=LARGE_SCALE_DOMAIN,
        regional_domain=REGIONAL_DOMAIN,
        chunk_days=TIME_REDUCTION_CHUNK_DAYS,
    )
    inventory.to_csv(INVENTORY_PATH, index=False)
    if ndjf_mean is None:
        raise RuntimeError('The climatology is not complete yet. The Drive inventory was saved; rerun this exact cell to resume.')
    local_mean_path = LOCAL_WORK_DIR / MEAN_CACHE_PATH.name
    ndjf_mean.to_netcdf(local_mean_path)
    shutil.copy2(local_mean_path, MEAN_CACHE_PATH)
    print('Complete NDJF mean saved:', MEAN_CACHE_PATH)

display(inventory['status'].value_counts().rename_axis('status').reset_index(name='calendar_months'))
print('Total hourly samples:', ndjf_mean.attrs['total_hourly_samples'])
display(ndjf_mean)

## Create slide-ready maps

The H and L are found from the **absolute seasonal mean MSLP** inside pre-declared search boxes, not from event anomalies. Vectors are actual mean 925-hPa winds. SST is shaded only in the Japan/East Sea display box, while 2-m air-temperature contours are shown across the regional map.

In [ ]:
if 'ndjf_mean' not in globals() or ndjf_mean is None:
    raise RuntimeError('Run the completed climatology-build cell first.')

def add_base_geography(ax, domain):
    ax.set_extent([domain.lon_min, domain.lon_max, domain.lat_min, domain.lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor='#d9d1bd', zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor='#d8e7ef', zorder=0)
    ax.add_feature(cfeature.BORDERS, linewidth=0.45, edgecolor='#374151', zorder=5)
    ax.coastlines(resolution='50m', linewidth=0.55, color='#1f2937', zorder=6)
    gridlines = ax.gridlines(draw_labels=True, linewidth=0.35, color='#64748b', alpha=0.45, linestyle='--')
    gridlines.top_labels = False
    gridlines.right_labels = False

def add_wind_vectors(ax, domain, step, scale, label, label_position=(0.98, 0.97)):
    u = subset_domain(ndjf_mean['u925_ms'], domain)
    v = subset_domain(ndjf_mean['v925_ms'], domain)
    quiver = ax.quiver(
        u.longitude.values[::step], u.latitude.values[::step],
        u.values[::step, ::step], v.values[::step, ::step],
        transform=ccrs.PlateCarree(), color='#111827', width=0.0023,
        headwidth=3.1, headlength=4.2, headaxislength=3.8, scale=scale, zorder=12,
    )
    ax.quiverkey(quiver, 0.10, 0.05, 10, '10 m s$^{-1}$', labelpos='E', coordinates='axes', color='#111827')
    ax.text(*label_position, label, transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.8, edgecolor='none'), zorder=20)

def pressure_label(ax, center, symbol, name, color):
    stroke = [pe.withStroke(linewidth=2.8, foreground='white')]
    ax.text(center['longitude_degE'], center['latitude_degN'], symbol, transform=ccrs.PlateCarree(),
            ha='center', va='center', fontsize=30, fontweight='bold', color=color, path_effects=stroke, zorder=22)
    ax.text(center['longitude_degE'], center['latitude_degN'] - 3.3, name, transform=ccrs.PlateCarree(),
            ha='center', va='top', fontsize=9, fontweight='bold', color=color, path_effects=stroke, zorder=22)

mslp = subset_domain(ndjf_mean['mslp_hpa'], LARGE_SCALE_DOMAIN)
high_center = find_pressure_center(mslp, SIBERIAN_HIGH_BOX, mode='high')
low_center = find_pressure_center(mslp, ALEUTIAN_LOW_BOX, mode='low')
center_summary = pd.DataFrame([
    {'feature': 'Siberian High', **high_center},
    {'feature': 'Aleutian Low', **low_center},
])
center_summary.to_csv(CENTER_SUMMARY_PATH, index=False)
display(center_summary.round(2))

# The final presentation map uses this wider terrain frame; it does not show the unused trans-Pacific panel.
terrain_cache = LOCAL_WORK_DIR / 'terrain_tiles_context'
terrain_rgb = build_terrain_relief(CONCEPT_MAP_DOMAIN, cache_dir=terrain_cache, zoom=6, output_width=2000, output_height=1000)
t2m = subset_domain(ndjf_mean['t2m_c'], REGIONAL_DOMAIN)
sst = subset_domain(ndjf_mean['sst_c'], REGIONAL_DOMAIN)
soj_mask = ((sst.longitude >= SEA_OF_JAPAN_DISPLAY_BOX.lon_min) & (sst.longitude <= SEA_OF_JAPAN_DISPLAY_BOX.lon_max)
            & (sst.latitude >= SEA_OF_JAPAN_DISPLAY_BOX.lat_min) & (sst.latitude <= SEA_OF_JAPAN_DISPLAY_BOX.lat_max))
sst_soj = sst.where(soj_mask)

fig_regional = plt.figure(figsize=(16, 8.4), constrained_layout=True)
ax_regional = fig_regional.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax_regional.set_extent([CONCEPT_MAP_DOMAIN.lon_min, CONCEPT_MAP_DOMAIN.lon_max, CONCEPT_MAP_DOMAIN.lat_min, CONCEPT_MAP_DOMAIN.lat_max], crs=ccrs.PlateCarree())
ax_regional.imshow(terrain_rgb, extent=[CONCEPT_MAP_DOMAIN.lon_min, CONCEPT_MAP_DOMAIN.lon_max, CONCEPT_MAP_DOMAIN.lat_min, CONCEPT_MAP_DOMAIN.lat_max],
                   origin='upper', transform=ccrs.PlateCarree(), zorder=0)
ax_regional.add_feature(cfeature.BORDERS, linewidth=0.45, edgecolor='#1f2937', zorder=6)
ax_regional.coastlines(resolution='50m', linewidth=0.65, color='#111827', zorder=7)
gridlines = ax_regional.gridlines(draw_labels=True, linewidth=0.35, color='white', alpha=0.55, linestyle='--')
gridlines.top_labels = False
gridlines.right_labels = False
sst_levels = np.arange(-1, 21, 1)
sst_mesh = ax_regional.contourf(sst_soj.longitude, sst_soj.latitude, sst_soj, levels=sst_levels, cmap='turbo', extend='both',
                                  alpha=0.88, transform=ccrs.PlateCarree(), zorder=3)
sst_colorbar = fig_regional.colorbar(sst_mesh, ax=ax_regional, pad=0.015, shrink=0.82)
sst_colorbar.set_label('NDJF mean SST (°C) — Sea of Japan only')
temperature_levels = np.arange(-30, 16, 5)
# Keep contours beneath the vectors, while retaining readable top-layer labels over the continental outflow corridor.
temperature_contours = ax_regional.contour(t2m.longitude, t2m.latitude, t2m, levels=temperature_levels, colors='#f8fafc',
                                           linewidths=1.15, transform=ccrs.PlateCarree(), zorder=9)
# The third placement follows the −15°C corridor so the nearby −10°C label has the same readable diagonal orientation.
temperature_label_positions = [(121.5, 52.0), (124.5, 49.0), (124.4, 45.2), (129.0, 39.5), (133.0, 36.5)]
temperature_labels = ax_regional.clabel(temperature_contours, manual=temperature_label_positions, fmt=lambda value: f'{value:+.0f}°C', fontsize=7, inline=True, inline_spacing=3)
# A second −10°C label goes on the eastern end of that contour, away from the Changbai Mountains label.
eastern_minus10_labels = ax_regional.clabel(temperature_contours, levels=[-10], manual=[(142.8, 45.0)], fmt=lambda value: f'{value:+.0f}°C', fontsize=7, inline=True, inline_spacing=3)
# Secondary labels identify the same actual 2-m-temperature contours over the Pacific side of the displayed regional field.
pacific_temperature_label_positions = [(147.0, 47.0), (151.5, 39.0), (150.0, 35.0)]
pacific_temperature_labels = ax_regional.clabel(temperature_contours, manual=pacific_temperature_label_positions, fmt=lambda value: f'{value:+.0f}°C', fontsize=7, inline=True, inline_spacing=3)
zero_temperature_labels = ax_regional.clabel(temperature_contours, levels=[0], manual=[(149.0, 43.5)], fmt=lambda value: f'{value:+.0f}°C', fontsize=7, inline=True, inline_spacing=3)
for temperature_label in [*temperature_labels, *eastern_minus10_labels, *pacific_temperature_labels, *zero_temperature_labels]:
    temperature_label.set_zorder(26)
    temperature_label.set_color('#111827')
    temperature_label.set_bbox(dict(boxstyle='round,pad=0.12', facecolor='white', edgecolor='none', alpha=0.80))
add_wind_vectors(ax_regional, CONCEPT_MAP_DOMAIN, step=5, scale=215, label='Actual NDJF mean 925-hPa wind', label_position=(0.74, 0.97))
pressure_label(ax_regional, high_center, 'H', 'Siberian High', '#0b5ea8')
pressure_label(ax_regional, low_center, 'L', 'Aleutian Low', '#b91c1c')
ax_regional.text(128.0, 42.1, 'Changbai\nMountains', transform=ccrs.PlateCarree(), ha='center', va='center', fontsize=10, color='white',
                 fontweight='bold', path_effects=[pe.withStroke(linewidth=2.5, foreground='black')], zorder=20)
ax_regional.text(136.0, 41.3, 'Sea of Japan', transform=ccrs.PlateCarree(), ha='center', va='center', fontsize=11, color='white',
                 fontweight='bold', path_effects=[pe.withStroke(linewidth=2.5, foreground='black')], zorder=20)
# Opaque, top-layer terrain key: it intentionally hides underlying wind arrows.
elevation_colors = np.array([[72, 123, 77], [115, 153, 83], [187, 175, 100], [235, 202, 107], [175, 118, 70], [138, 88, 67]]) / 255.0
elevation_cmap = LinearSegmentedColormap.from_list('jpcz_terrain_key', elevation_colors)
elevation_ax = ax_regional.inset_axes([0.898, 0.075, 0.088, 0.285], zorder=50)
elevation_ax.set_facecolor((1, 1, 1, 0.94))
elevation_ax.patch.set_alpha(0.94)
elevation_ax.set_xlim(0, 1)
elevation_ax.set_ylim(0, 1)
elevation_ax.set_xticks([])
elevation_ax.set_yticks([])
for spine in elevation_ax.spines.values():
    spine.set_linewidth(0.9)
    spine.set_edgecolor('#111827')
elevation_ax.text(0.50, 0.93, 'Elevation (m)', ha='center', va='top', fontsize=8, fontweight='bold', color='#111827')
elevation_ramp = np.linspace(0, 1, 256).reshape(-1, 1)
elevation_ax.imshow(elevation_ramp, extent=[0.66, 0.88, 0.13, 0.76], origin='lower', aspect='auto', cmap=elevation_cmap, zorder=52)
elevation_ax.add_patch(Rectangle((0.66, 0.13), 0.22, 0.63, fill=False, linewidth=1.0, edgecolor='#111827', zorder=53))
for elevation_m, y_position in [(3000, 0.70), (2000, 0.51), (1000, 0.32), (0, 0.13)]:
    elevation_ax.text(0.60, y_position, f'{elevation_m}', ha='right', va='center', fontsize=7, color='#111827', zorder=54)
ax_regional.set_title(f'NDJF {min(CLIMATOLOGY_YEARS)}–{max(CLIMATOLOGY_YEARS)} mean terrain, SST, 2-m air temperature, and 925-hPa wind', fontsize=14, pad=14)
ax_regional.text(0.01, 0.01, 'Terrain background: Mapzen / AWS Terrain Tiles. White contours: mean 2-m air temperature. H: Siberian High.', transform=ax_regional.transAxes, fontsize=8,
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.55, edgecolor='none'), color='white', zorder=20)
regional_local_path = LOCAL_FIGURE_DIR / REGIONAL_FIGURE_PATH.name
fig_regional.savefig(regional_local_path, dpi=220, bbox_inches='tight')
shutil.copy2(regional_local_path, REGIONAL_FIGURE_PATH)
plt.show()

print('Saved expanded terrain/SST/H figure:', REGIONAL_FIGURE_PATH)

## Presentation wording

**Winter setting.** The blue H and red L mark the objectively located NDJF Siberian-High and Aleutian-Low pressure centers. The actual mean 925-hPa vectors show the resulting predominantly northwesterly continental outflow toward the Japan/East Sea.

**Regional setting.** As the cold continental flow reaches the Changbai/coastal terrain and then the relatively warm Sea of Japan, its path is modified by the terrain and air–sea temperature contrast. This map is a seasonal background; it does not claim that every JPCZ event has the same instantaneous wind pattern.

Use this single expanded terrain–SST–low-level-flow panel for the pressure-gradient / monsoon and regional-terrain context.